# Shoreline Projection Methodology (Updated)

This document describes the methods implemented in `trend_uncertainty.py` to derive shoreline projections from satellite-derived shoreline positions, beach slope, and sea-level-rise (SLR) projections.

The workflow combines deterministic process understanding (Bruun-style SLR response) with stochastic sampling (bootstrap trends and Monte Carlo uncertainty propagation).

## 1. Input Data and Preprocessing

### 1.1 Satellite shoreline time series
For each CoastSat site and transect, shoreline position observations are loaded and converted to decimal years:

$$
t = Y + \frac{DOY - 1}{365.25}
$$

where $Y$ is calendar year and $DOY$ is day-of-year.

Missing shoreline observations are removed per transect before trend analysis.

### 1.2 Most recent continuous segment
The function `filter_to_longest_consecutive_year_run` splits each transect series at gaps longer than a threshold (configured in months) and retains the most recent segment whose span exceeds a minimum length:

- gap threshold: `max_gap_months` (used as 10 months in the main loop)
- minimum span: `min_span_years` (set from `loess_window`)

Transects without a valid recent segment are skipped.

## 2. Smoothing and Trend Extraction

### 2.1 LOESS smoothing
A LOESS smoother is applied to the filtered shoreline time series to reduce short-term noise while preserving multiyear structure. The smoothing fraction is dynamic:

$$
frac = clip\left(\frac{W}{T_{span}},\ 0.15,\ 0.9\right)
$$

where $W$ is the configured LOESS window (`loess_window`) and $T_{span}$ is the filtered record duration in years.

### 2.2 Block bootstrap of trend slopes
From the LOESS-smoothed series, trend uncertainty is estimated via block bootstrap (`block_bootstrap_slopes`):

1. Randomly choose a valid start time.
2. Build a fixed-duration window (`block_years`).
3. Interpolate the endpoint if needed so each realization spans exactly `block_years`.
4. Fit a linear model in that window and keep slope $r_i$ (m/yr).

Per realization, the local slope is:

$$
r_i = \hat{a}_1\ \text{from}\ y = \hat{a}_0 + \hat{a}_1 t
$$

repeated for `n_boot` realizations to form an empirical slope pool.

An optional recent-only slope pool is also computed from the latest years (`recent_trend_window_years`) and can be blended into the first projection segment.

## 3. Shoreline-Change Formulation

### 3.1 SLR-driven shoreline response
The SLR contribution follows a Bruun-style scaling with sign convention of retreat as negative shoreline change:

$$
\Delta y_{SLR} = -\frac{c}{\tan\beta}\Delta S
$$

where:
- $c$ is a calibration factor (used as 1.0),
- $\tan\beta$ is beach slope (with uncertainty sampled in Monte Carlo),
- $\Delta S$ is sea-level rise relative to the projection baseline.

### 3.2 Combined SLR + trend over segmented horizon
The projection horizon $dt$ is split into segments $\Delta t_k$ (typically 10-year segments with optional decade alignment).

For each Monte Carlo realization and segment:

$$
\Delta y_k = -\frac{c}{\tan\beta}\Delta S\left(\frac{\Delta t_k}{dt}\right) + r_k\Delta t_k
$$

and total projected change is:

$$
\Delta y_{total} = \sum_{k=1}^{K} \Delta y_k
$$

where $r_k$ is a sampled trend rate for segment $k$.

## 4. Uncertainty Representation

The Monte Carlo function `mc_shoreline_change` propagates uncertainty from three sources:

1. **Trend-rate uncertainty** from bootstrap slope pools.
2. **Beach-slope uncertainty** sampled as uniform perturbation around nominal slope:

$$
\tan\beta^{(j)} \sim U(0.8\tan\beta_0,\ 1.2\tan\beta_0)
$$

3. **SLR uncertainty** from NZ SeaRise quantiles (17th, 50th, 83rd).

If quantiles are provided, a Gaussian approximation is fitted:

$$
q_p = \mu + \sigma z_p
$$

with $p=0.17,0.50,0.83$, $\mu=q_{50}$, and:

$$
\sigma_{low} = \frac{q_{50}-q_{17}}{z_{0.83}}, \quad
\sigma_{high} = \frac{q_{83}-q_{50}}{z_{0.83}}, \quad
\sigma = \frac{\sigma_{low}+\sigma_{high}}{2}
$$

then:

$$
\Delta S^{(j)} \sim \mathcal{N}(\mu,\sigma)
$$

Otherwise, deterministic $\Delta S$ is used.

## 5. Projection Segmentation and First-Segment Recent Weighting

### 5.1 Segment construction
`build_projection_segment_durations` creates segment durations that sum to $dt$.

When decade alignment is enabled, the first segment ends at the next decade boundary, then fixed-length segments follow. For example, if projection starts in 2025 with 10-year segments, the first segment is 5 years (to 2030), then 10-year blocks.

### 5.2 Recent-trend blend in first segment
If recent trend samples are available, only segment 1 can blend historical and recent slope pools with weight $w$ (`first_segment_recent_weight`):

$$
r_1 \sim
\begin{cases}
R_{recent}, & \text{with probability } w \\ 
R_{historic}, & \text{with probability } 1-w
\end{cases}
$$

For segments $k>1$, trend samples are taken from the historical pool only.

## 6. Outputs and Summary Statistics

For each site/transect, the workflow stores:

- segment-wise total change realizations ($\Delta y_k$),
- segment-wise SLR-only and trend-only components,
- sampled trend rates per segment,
- summary statistics of total change:

$$
P05,\ Median,\ P95\ of\ \Delta y_{total}
$$

- projection time series exported to CSV for downstream dashboard plotting (mean, q05, q95 for total, SLR-only, and trend-only pathways).

Diagnostic plots are also generated per transect (original series, LOESS-smoothed series, and observed + projected envelopes).

## 7. Assumptions and Practical Notes

- The Bruun-style scaling is treated as a first-order shoreline response model.
- Alongshore processes and sediment-budget complexities are not explicitly simulated in `trend_uncertainty.py`.
- Trend uncertainty is represented empirically by block-bootstrapped local linear slopes from smoothed observations.
- SLR uncertainty is represented either deterministically or by a Gaussian fit to NZ SeaRise quantiles.
- Reported uncertainty bands therefore combine process sensitivity (slope), forcing uncertainty (SLR), and data-derived trend variability.